# Network analysis

In [ ]:
# libraries
import pandas as pd
import os
import networkx as nx
import math

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import contextily as cx

# local functions
from utils import import_matrix
from assign import AONloading, Assignment # this module contains the AONloading function from tutorial 3 and the Assignment function you wrote for tutorial 3 homework

pd.options.mode.chained_assignment = None 

### Loading data

In [ ]:
root    = './'
city    = 'Dresden'

In [ ]:
#-------------------- Network --------------------

# Importing the networks into a Pandas dataframe consists of a single line of code
# but we can also make sure all headers are lower case and without trailing spaces

netfile = os.path.join(root, city, 'net.csv')
network = pd.read_csv(netfile)#, header=7)

trimmed= [s.strip().lower() for s in network.columns]
network.columns = trimmed


In [ ]:
#-------------------- Network attributes --------------------

linkindexordering = list(zip(network['from'], network['to']))
edge_attr = {}
mlt = False
for row in network.itertuples():
    edge_attr[(row[1], row[2])] = {'capacity':row[3], 'length':row[4], 'FFT':row[5], 'alpha':row[6],
                                    'beta':row[7], 'type':row[10], 'flow':0, 'cost':row[5], 'Oflow':None}

edge = [(row[1], row[2]) for row in network.itertuples()]

In [ ]:
#-------------------- Graph --------------------

NetGraph = nx.DiGraph()
NetGraph.add_edges_from(edge)
nx.set_edge_attributes(NetGraph, edge_attr)


In [ ]:
demandfile = os.path.join(root, city, 'ODDemand.csv')
odDemands = pd.read_csv(demandfile, header=0)
ODdemand = {(int(row['Origin']), int(row['Destination'])):int(row['Demand']) for index, row in odDemands.iterrows()}

In [ ]:
#-------------------- Zone centroids --------------------

centroidspath = os.path.join(root, city, 'centroid.csv')
centroids = pd.read_csv(centroidspath, header=0)

if 'fishnetID' in centroids.columns:
    centroids.rename(columns={'fishnetID': 'ZoneID'}, inplace=True)

zone2cent = {zone: [row['osmid'] for index, row in centroids[centroids['ZoneID'] == zone].iterrows()] for zone in centroids['ZoneID']}


### Noise emission estimation

In [ ]:
# Free flow OD travel times

FreeflowGraph = NetGraph.copy()

SPTT, x_bar, spedges, FFODTT = AONloading(FreeflowGraph, zone2cent, ODdemand, computesptt=True)

In [ ]:
# OD travel times with noise

accuracy, maxiter = 0.1, 15

G, TSTT, SPEdges, EODTT = Assignment(NetGraph, zone2cent, ODdemand, accuracy, maxiter)



In [ ]:
def calculate_noise(data):
    L0 = 37.3  # Base noise level for light vehicles (can be adjusted) default 50
    k = 30   # Speed correction coefficient we can also use the standard formula
    distance = 25  # Reference distance in meters default 15

    flow = data['flow']  # Flow (vehicles per hour)
    speed = (data['length'] / (data['cost'] * 60)) * 3.6  # Convert m/s to km/h
    speed = min(speed, 100)  # Cap at 100 km/h for realistic values
    Lcar= 27.7 + 10 * np.log10(1 + (0.02 * speed)**3)
    Lhg= 23.1 + 12.5 * np.log10(speed) # speed should be for heavy goods vehicle instead
    f2= 10**(0.1 * (Lhg - Lcar)) 
    k2=2
    speedC= Lcar - L0 + 10 * np.log10((100 + (f2-1)) * k2) / (100 + 8.23 * k2)
    if flow == 0:
        return 0  # No noise if no traffic flow
   
    Noiseavg= L0 + 10 * np.log10(flow * (1 + 0.082 * 3))
    Leq= (Noiseavg+speedC) #*data['length']
   
    return Leq

### Compute Noise for each edge

In [ ]:
edge_noise = pd.DataFrame(columns=['from', 'to', 'noise'])
for u, v, data in G.edges(data=True):
    G.edges[(u, v)]['noise'] = calculate_noise(data)
    edge_noise = pd.concat([edge_noise, pd.DataFrame.from_records([{'from': u, 'to': v, 'noise': data['noise']}])])


In [ ]:
edges = gpd.read_file(os.path.join(root, city, 'shapefiles/edges.geojson')) # for geojson files use edges.geojson
zones = gpd.read_file(os.path.join(root, city, 'shapefiles/aggrid_polygon.geojson')) # for geojson files use aggrid_polygon.geojson
districts= gpd.read_file(os.path.join(root, city, 'shapefiles/dresden.geojson'))
if 'fishnetID' in zones.columns:
    zones.rename(columns={'fishnetID': 'ZoneID'}, inplace=True)

#districts.columns

#### Aggregate noise emmissions by Zone

In [ ]:
edges = edges.merge(edge_noise, on=['from', 'to'], how='left')
#edges=edges[(edges['edge_noise']>60)]
edges = gpd.GeoDataFrame(edges, geometry='geometry')

edge_noise_gpd = gpd.sjoin(districts, edges, how='left')[['name', 'length', 'noise']]
edge_noise_gpd = edge_noise_gpd[(edge_noise_gpd['noise']>60)]
edge_noise_gpd['noise'] = edge_noise_gpd['noise']*edge_noise_gpd['length']
district_noise = edge_noise_gpd.groupby('name').sum()
district_noise ['noise']= district_noise ['noise']/district_noise['length']


### Gini Index for Noise

In [ ]:
def gini(x):
    total = 0
    for i, xi in enumerate(x[:-1], 1):
        total += np.sum(np.abs(xi - x[i:]))
    return total / (len(x)**2 * np.mean(x))

In [ ]:
gini_em = gini(district_noise['noise'])

print(f"gini_district_noise {gini_em} ")

In [ ]:
import matplotlib.cm as cm
fig, ax = plt.subplots(figsize=(10, 10))
edges.plot(ax=ax, column='noise', cmap='coolwarm', legend=False) #, vmin=0, vmax = 15
plt.text(0.07, 0.65, 'Bundesautobahn 4',
         transform=ax.transAxes,  # This makes coordinates relative to axes (0 to 1)
         fontsize=10,
         color='black',
         fontstyle='italic',
         fontweight='bold',
         bbox=dict(facecolor='none', edgecolor='none', alpha=0.7)) 
cx.add_basemap(ax, crs=edges.crs.to_string(), source=cx.providers.OpenStreetMap.Mapnik, zoom = 12, alpha=0.6)
sm = cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=0, vmax=75))
# sm.set_array([])
cbar = fig.colorbar(sm,ax=ax, fraction=0.04, pad=0.02)
ax.set_title("Link-wise noise emissions", fontsize=16, fontweight='bold')
ax.set_xticks([])  # Remove x-axis labels
ax.set_yticks([])  # Remove y-axis labels
ax.set_xticklabels([])  # Remove x-axis tick labels
ax.set_yticklabels([])  # Remove y-axis tick labels


In [ ]:
import matplotlib.colors as mcolors

# Step 1: Filter and bin the noise data
#edges_filtered = edges[edges['noise'] > 0].copy()

# Define discrete bins and labels (adjust these thresholds as needed)
bins = [0, 55, 60, 65, 70, 75, 100]
labels = ['<50 dB', '50–54 dB', '55–59 dB', '60–64 dB', '65–70 dB','>70 dB']

# Create a new categorical column for plotting
edges['noise_range'] = pd.cut(edges['noise'], bins=bins, labels=labels, include_lowest=True)

# Step 2: Plot with categorical colormap
fig, ax = plt.subplots(figsize=(10, 10))

edges.plot(
    ax=ax,
    column='noise_range',
    cmap='RdYlBu_r',   # A good discrete color map (can try 'Set1', 'tab10', etc.)
    legend=True,
    linewidth=1
)

# Add basemap
cx.add_basemap(ax, crs=edges.crs.to_string(), source=cx.providers.OpenStreetMap.Mapnik, zoom=12, alpha=0.6)

# Clean the plot
ax.set_title("Link-wise excess noise", fontsize=16, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])

In [ ]:
from matplotlib.colors import ListedColormap

# Example color list matching the visual style
colors = [
    "#c6d9d0",  # > 50–54 dB (light cyan)
    "#f2eec1",  # 55–59 dB (beige/yellow)
    "#f6c181",  # 60–64 dB (light orange)
    "#cc6633",  # 65–69 dB (brownish orange)
    "#990066",  # 70–74 dB (purple)
    "#4d0039"   # >75 dB (dark violet)
]

# Define noise bins and labels
bins = [40, 45, 50, 55, 60, 65, 100]
labels = [
    "> 50–54 dB(A)",
    "> 55–59 dB(A)",
    "> 60–64 dB(A)",
    "> 65–69 dB(A)",
    "> 70–74 dB(A)",
    "> 75 dB(A)"
]

# Categorize the data
#edges_filtered = edges[edges['noise'] > 0].copy()
edges["noise_band"] = pd.cut(edges["noise"], bins=bins, labels=labels, include_lowest=True)

# Create the custom colormap
cmap = ListedColormap(colors)

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
edges.plot(
    ax=ax,
    column="noise_band",
    cmap=cmap,
    legend=True,
    linewidth=1.2
)

# Add basemap
cx.add_basemap(ax, crs=edges.crs, source=cx.providers.OpenStreetMap.Mapnik, alpha=0.6, zoom=12)

# Style
ax.set_title("Road Traffic Noise – Day-Evening-Night Indicator", fontsize=16, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])
ax.set_axis_off()

In [ ]:
print(edges.crs)


In [ ]:
edges_export = edges[['from', 'to', 'noise', 'length', 'geometry']].copy()
edges_export['noise'] = edges_export['noise'].fillna(0)


In [ ]:
output_path = os.path.join(root, city, "link_noise.geojson")

edges_export.to_file(output_path, driver="GeoJSON")

print("GeoJSON with noise exported successfully!")


In [ ]:
edges_export['noise_length_weighted'] = edges_export['noise'] * edges_export['length']


In [ ]:
output_path = os.path.join(root, city, "link_weighted_noise.geojson")

edges_export.to_file(output_path, driver="GeoJSON")

### CNOSSOS-EU method for Noise attenuation

In [ ]:
distances = [0, 10, 25, 50, 100, 200]
"""
Geometrical spreading/divergence noise at a height of 4m according to EU guidelines     
"""
ring_layers = []

for d_outer, d_inner in zip(distances[1:], distances[:-1]):
    temp = edges.copy()

    outer = temp.geometry.buffer(d_outer)
    inner = temp.geometry.buffer(d_inner)

    temp['geometry'] = outer.difference(inner)
    temp['distance_band'] = f"{d_inner}-{d_outer} m"

    # Apply decay at mid-distance 
    d_mid = (d_outer + d_inner) / 2
    temp['noise_d'] = temp['noise'].apply(
        lambda L: L - 20 * np.log10(d_mid / 25)
    )

    ring_layers.append(temp)
    


In [ ]:
rings = gpd.GeoDataFrame(
    pd.concat(ring_layers, ignore_index=True),
    crs=edges.crs
)
rings['noise_d'] = rings['noise_d'].clip(lower=0)


In [ ]:
print(rings.crs)


In [ ]:
rings['geometry'] = rings['geometry'].buffer(0)


In [ ]:
rings_export = rings[['from', 'to', 'distance_band', 'noise_d', 'geometry']].copy()


In [ ]:
rings_export['noise_ref'] = rings['noise']


In [ ]:
output_path = os.path.join(root, city, "noise_decay_bands.geojson")

rings_export.to_file(output_path, driver="GeoJSON")


In [ ]:
rings_dissolved = rings_export.dissolve(by='distance_band', aggfunc='mean')
rings_dissolved.to_file("noise_decay_dissolved.geojson", driver="GeoJSON")
